# riss_drucker M0 smoke test

Thin driver (README section 3): all logic lives in the library.

1. Probe the environment (fails fast if a build tool is missing).
2. Build the wheel offline from the vendored source, or install a prebuilt wheel from a dataset.
3. Check that the scalar and SIMD kernels agree, eigh works on every built backend, and write a run record.

On Kaggle, attach the source as a dataset (stamp it first with `git rev-parse HEAD > GIT_COMMIT`), or set `RISS_DRUCKER_SRC`. To reuse a wheel from an earlier session, set `PREBUILT_WHEEL_DIR`.

In [ ]:
import os, sys, json, glob, subprocess
from pathlib import Path

def find_repo():
    if os.environ.get("RISS_DRUCKER_SRC"):
        return Path(os.environ["RISS_DRUCKER_SRC"])
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "scripts" / "kaggle_build.sh").is_file():
            return cand
    hits = glob.glob("/kaggle/input/**/scripts/kaggle_build.sh", recursive=True)
    if hits:
        return Path(hits[0]).parent.parent
    raise FileNotFoundError("riss-drucker source not found; set RISS_DRUCKER_SRC")

REPO = find_repo()
WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else REPO / "build-kaggle"))
PREBUILT_WHEEL_DIR = os.environ.get("PREBUILT_WHEEL_DIR", "")   # e.g. /kaggle/input/riss-drucker-wheel
BACKEND_FLAGS = []   # e.g. ["--lapack"] if env_probe reports lapack_linkable
print(REPO, WORK)

In [ ]:
args = ["--runtime-only"] if PREBUILT_WHEEL_DIR else []
r = subprocess.run([sys.executable, str(REPO / "scripts" / "env_probe.py"), *args, "--out", str(WORK / "env_probe.json")],
                   capture_output=True, text=True)
probe = json.loads(r.stdout[: r.stdout.rindex("}") + 1])
print({k: probe[k] for k in ("ok", "problems", "kaggle")})
print("cpu:", probe["cpu"].get("model_name"), probe["cpu"]["flags"])
print("cores:", probe["cpu"]["logical_cores_affinity"], " mem:", probe["memory"], " gpu:", probe["gpu"])
print("blas:", probe["blas"])
assert probe["ok"], probe["problems"]

In [ ]:
env = {**os.environ, "RISS_DRUCKER_WORK": str(WORK), "PYTHON": sys.executable}
cmd = ["bash", str(REPO / "scripts" / "kaggle_build.sh"), *BACKEND_FLAGS]
if PREBUILT_WHEEL_DIR:
    cmd += ["--wheel", PREBUILT_WHEEL_DIR]
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
r.check_returncode()

In [ ]:
import importlib, numpy as np
import riss_drucker as rd
importlib.reload(rd)

print(json.dumps(rd.build_info(), indent=1))
print("SIMD:", rd.simd_level_active(), "available:", rd.simd_levels_available())
print("linalg backends:", rd.linalg_backends())
print("runtime config:", rd.runtime_config().as_dict())

In [ ]:
# Scalar vs dispatched SIMD agreement (README section 4: numerical safety).
rng = np.random.default_rng(0)
for n in (7, 1000, 100_003):
    x, y = rng.standard_normal(n), rng.standard_normal(n)
    ref = rd.dot(x, y, level="scalar")
    tol = 4 * (n + 1) * np.finfo(float).eps * (np.abs(x * y).sum() + 1)
    for lvl in rd.simd_levels_available():
        dev = abs(rd.dot(x, y, level=lvl) - ref)
        assert dev <= tol, (n, lvl, dev, tol)
        print(f"dot n={n:>7} {lvl:>7}: |dev|={dev:.2e} (tol {tol:.1e})")

# Dense eigh on every built backend.
q, _ = np.linalg.qr(rng.standard_normal((200, 200)))
lam = np.linspace(-1, 1, 200)
a = (q * lam) @ q.T
for b in rd.linalg_backends():
    w, v = rd.eigh(a, backend=b)
    print(f"eigh[{b}] max|w - lam| = {np.abs(w - lam).max():.2e}, residual = {np.abs(a @ v - v * w).max():.2e}")
    assert np.abs(w - lam).max() < 1e-12

In [ ]:
rec_path = rd.write_run_record(WORK / "run_record_smoke.json", seeds={"numpy": 0},
                               env_probe=json.loads((WORK / "env_probe.json").read_text()))
print("run record:", rec_path)
print("M0 smoke test passed.")